# Bài 15: Biến đổi Fourier và Lọc trong miền tần số

Mọi bộ lọc từ đầu đến giờ đều làm việc trực tiếp trên các vùng lân cận điểm ảnh. Biến đổi Fourier (Fourier transform) mang lại một góc nhìn hoàn toàn khác: bất kỳ ảnh nào cũng có thể được viết lại thành tổng của các vân sóng hình sin (sinusoidal gratings) ở các tần số và hướng khác nhau. Khi đó, việc lọc có thể thực hiện bằng cách trực tiếp tăng cường hoặc triệt tiêu các tần số cụ thể &mdash; và nhờ **định lý tích chập (convolution theorem)**, điều này hóa ra về mặt toán học chính là phép tích chập trong miền không gian, chỉ là nhìn từ một góc độ khác.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Khởi động 1D: tín hiệu là tổng các hình sin

Bất kỳ tín hiệu nào cũng có thể được xấp xỉ bằng cách cộng các sóng sin và cosin ở các tần số khác nhau. Ví dụ, sóng vuông là tổng của các sóng sin ở tần số tăng dần (các họa âm lẻ, biên độ giảm dần). Biến đổi Fourier là công cụ đi theo chiều ngược lại: cho một tín hiệu, nó cho bạn biết chính xác những tần số nào (và mỗi tần số bao nhiêu) có mặt.

In [ ]:
t = np.linspace(0, 1, 500, endpoint=False)
square_wave = np.sign(np.sin(2 * np.pi * 5 * t))

approx = np.zeros_like(t)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for n_harmonics in [1, 9]:
    approx = sum((4 / (np.pi * k)) * np.sin(2 * np.pi * 5 * k * t) for k in range(1, n_harmonics + 1, 2))
    axes[0 if n_harmonics == 1 else 1].plot(t, square_wave, '--', color='gray', label='true square wave')
    axes[0 if n_harmonics == 1 else 1].plot(t, approx, label=f'{n_harmonics} harmonic(s)')
    axes[0 if n_harmonics == 1 else 1].legend(fontsize=8)
    axes[0 if n_harmonics == 1 else 1].set_title(f'{n_harmonics} harmonic(s)', fontsize=9)
plt.tight_layout()
plt.show()

Ảnh động này xây dựng sóng vuông bằng cách thêm dần từng cặp họa âm lẻ, cho thấy tổng riêng phần hội tụ về sóng vuông thực sự, trong khi một độ vọt lố cố định ("rung — ringing") gần mỗi bước nhảy không bao giờ biến mất hoàn toàn — đó là **hiện tượng Gibbs (Gibbs phenomenon)**. Lý do: tái tạo chính xác một bước nhảy tức thời đòi hỏi *vô số* tần số, nên việc cắt bớt xuống một số hữu hạn họa âm thực chất là một ngưỡng cắt cứng trong miền tần số — một số tần số được giữ nguyên vẹn, phần còn lại bị loại bỏ hoàn toàn. Chính ngưỡng cắt cứng đó tạo ra gợn sóng, một hiện tượng sẽ xuất hiện lại ở phần sau của bài mỗi khi các tần số bị cắt đột ngột thay vì giảm dần một cách mượt mà.

<img src="../animations/fourier_synthesis_demo.gif" alt="Ảnh động tổng Fourier riêng phần thêm dần từng họa âm lẻ, hội tụ về sóng vuông với độ vọt lố Gibbs dai dẳng gần các bước nhảy" style="display: block; margin: 0 auto; max-width: 680px;">

### Các cặp Fourier: sắc nét ở miền này, trải rộng ở miền kia

Sóng vuông ở trên chỉ dùng biến đổi Fourier 1D một cách không chính thức, như một tổng các hình sin. Chính xác hơn, biến đổi Fourier biến *bất kỳ* tín hiệu nào thành một hàm theo tần số liên tục (gọi là **phổ (spectrum)** của nó), và hai hàm này liên kết với nhau thành một **cặp Fourier (Fourier pair)**: gọn và sắc nét ở miền này luôn tương ứng với trải rộng ở miền kia. Ví dụ rõ nhất: một xung hộp có cạnh sắc và hữu hạn trong không gian, nhưng biến đổi của nó là một hàm **sinc** kéo dài — và dao động — mãi mãi (hộp ↔ sinc). Một trường hợp còn đơn giản hơn là sóng sin thuần túy, trải rộng mãi mãi trong không gian, có biến đổi Fourier là hai gai nhọn trong miền tần số (sin ↔ gai) — ta sẽ thấy điều này ngay sau đây.

In [ ]:
x = np.linspace(-4, 4, 1000)
box = ((x > -0.5) & (x < 0.5)).astype(float)

f = np.linspace(-8, 8, 1000)
sinc_transform = np.sinc(f)  # np.sinc(x) = sin(pi*x)/(pi*x), exactly the FT of a unit-width box

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(x, box)
axes[0].set_title('Box pulse (sharp, compact in space)')
axes[0].set_xlabel('x')
axes[1].plot(f, sinc_transform)
axes[1].set_title('Its Fourier transform: a sinc (spread out, oscillating)')
axes[1].set_xlabel('frequency')
plt.tight_layout()
plt.show()

## Biến đổi Fourier 2D của một ảnh

Biến đổi Fourier rời rạc (discrete Fourier transform — DFT) 2D của một ảnh phân rã nó thành các vân sóng hình sin 2D ở mọi tổ hợp tần số ngang và dọc. Thuật toán tính DFT là biến đổi Fourier nhanh (fast Fourier transform — FFT) (`np.fft.fft2`), cho ra một mảng 2D giá trị phức (thực + ảo). Tương tự gradient của ảnh, ta thường xem **độ lớn** (mỗi tần số có mặt bao nhiêu) và **hướng** của biến đổi Fourier. Hai mẹo phổ biến để trực quan hóa: hiển thị độ lớn theo thang logarit, và dịch số hạng tần số không (độ sáng trung bình) từ góc trên-trái về chính giữa.

In [ ]:
def show_spectrum(image, ax, title):
    F = np.fft.fftshift(np.fft.fft2(image)) # compute discrete Fourier transform (DFT), and shift DC value to center
    magnitude = np.log1p(np.abs(F)) # display logarithm of magnitude
    ax.imshow(magnitude, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    return F

img = np.zeros((200, 200), dtype=np.float64)
cv2.rectangle(img, (60, 60), (140, 140), 200, -1)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Image')
axes[0].axis('off')
show_spectrum(img, axes[1], 'Log-magnitude spectrum')
plt.tight_layout()
plt.show()

Hình chữ thập sáng đi qua tâm đến từ các cạnh ngang và dọc sắc nét của hình vuông — một biên bậc gắt chứa năng lượng ở *mọi* tần số dọc theo phương vuông góc với nó, đó là lý do một hình có cạnh sắc lại có phổ trải rộng đến vậy (nguyên lý cặp Fourier ở trên: sắc nét ở miền này, trải rộng ở miền kia).

### Một vân sóng thể hiện tần số của nó trực tiếp thành đỉnh phổ

Như đã báo trước ở trên (sin ↔ gai), một vân sóng hình sin thuần túy là phép thử đơn giản nhất có thể: phổ của nó phải (gần như) là một cặp chấm sáng duy nhất, cách tâm một khoảng bằng tần số của nó, theo hướng vuông góc với các sọc.

Tại sao lại là một *cặp*, mà không phải một chấm? Theo công thức Euler, sóng sin giá trị thực là sự pha trộn với lượng bằng nhau của hai hàm mũ phức: $\sin(2\pi f x) = \frac{1}{2i}\left(e^{i2\pi fx} - e^{-i2\pi fx}\right)$. Biến đổi Fourier của hàm mũ thứ nhất là một gai ở tần số $f$, còn của hàm kia là ở $-f$. Do đó, một hình sin thực tạo ra hai gai ở $+f$ và $-f$, đối xứng qua tâm tần số không (DC). Thực tế, với *bất kỳ* ảnh giá trị thực nào, độ lớn Fourier của nó đều đối xứng: $|F(-f)|=|F(f)|$.

In [ ]:
size = 100
x = np.arange(size)
cycles_across_image = 12
grating = np.tile(np.sin(2 * np.pi * cycles_across_image * x / size), (size, 1))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(grating, cmap='gray')
axes[0].set_title(f'Grating, {cycles_across_image} cycles across')
axes[0].axis('off')
F_grating = show_spectrum(grating, axes[1], 'Spectrum')
plt.tight_layout()
plt.show()

peak = np.unravel_index(np.argmax(np.abs(F_grating)), F_grating.shape)
print(f'brightest spectrum point at offset {peak[1] - size // 2} from center '
      f'(grating frequency was {cycles_across_image})')

Ảnh động này xoay vân sóng qua các hướng khác nhau và cho thấy phổ của nó cập nhật trực tiếp &mdash; cặp đỉnh sáng luôn nằm theo hướng vuông góc với các sọc, xoay đồng bộ với chúng.

<img src="../animations/grating_spectrum_demo.gif" alt="Ảnh động vân sóng hình sin xoay qua các hướng cạnh phổ độ lớn FFT của nó, với cặp đỉnh xoay vuông góc với các sọc" style="display: block; margin: 0 auto; max-width: 680px;">

Ảnh động bên dưới thay vào đó quét tần số của vân sóng (số chu kỳ trên toàn ảnh) từ thấp đến cao rồi quay lại (trong khi giữ hướng cố định). Cặp đỉnh phổ di chuyển thẳng ra xa tâm khi tần số tăng, và quay vào trong (về phía gốc) khi tần số giảm &mdash; với hướng không đổi.

<img src="../animations/frequency_spectrum_demo.gif" alt="Ảnh động vân sóng hình sin ở hướng cố định với tần số quét từ thấp đến cao rồi quay lại, cạnh phổ độ lớn FFT của nó, cho thấy cặp đỉnh di chuyển ra ngoài và vào trong dọc theo một hướng cố định" style="display: block; margin: 0 auto; max-width: 680px;">

## Lọc bằng cách chỉnh sửa phổ

**Bộ lọc thông thấp lý tưởng (ideal low-pass filter)** chỉ giữ lại các tần số bên trong một hình tròn quanh tâm (làm mờ); **bộ lọc thông cao lý tưởng (ideal high-pass filter)** chỉ giữ lại các tần số bên ngoài hình tròn đó (tăng cường biên).

In [ ]:
def circular_mask(shape, radius, keep_inside=True):
    h, w = shape
    yy, xx = np.mgrid[:h, :w]
    dist = np.sqrt((yy - h / 2)**2 + (xx - w / 2)**2)
    inside = dist <= radius
    return inside if keep_inside else ~inside

photo = np.zeros((200, 200), dtype=np.float64)
cv2.rectangle(photo, (40, 40), (120, 120), 200, -1)
cv2.circle(photo, (150, 150), 30, 120, -1)

low_pass_mask = circular_mask(photo.shape, radius=15, keep_inside=True)
high_pass_mask = circular_mask(photo.shape, radius=15, keep_inside=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
titles = ['Ideal low-pass mask' + chr(10) + '(frequency domain)',
          'Ideal high-pass mask' + chr(10) + '(frequency domain)']
for ax, im, title in zip(axes, [low_pass_mask, high_pass_mask], titles):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    # thin border so the mask's full extent is visible even when it's mostly
    # white (the high-pass mask) and would otherwise blend into the page;
    # clip_on=False so the line isn't half-clipped by the axes edge
    h, w = im.shape
    ax.add_patch(plt.Rectangle((-0.5, -0.5), w, h, fill=False, edgecolor='black',
                                linewidth=0.8, clip_on=False))
plt.tight_layout()
plt.show()

Khi ảnh đã được biến đổi sang miền tần số, ta có thể lọc nó bằng cách che trực tiếp các tần số, rồi biến đổi trở lại bằng biến đổi ngược. Vì mặt nạ có cùng kích thước với ảnh, ta chỉ cần nhân từng phần tử trong miền tần số. Hãy để ý một tác dụng phụ trong kết quả bên dưới: cắt tần số đột ngột gây ra hiện tượng rung mờ gần các cạnh sắc — chính là hiện tượng rung do ngưỡng cắt cứng đã thấy trong ví dụ sóng vuông ở trên, giờ xuất hiện lại dưới dạng các gợn sóng tỏa ra từ các cạnh sắc của bức ảnh. Lý do cũng chính là cặp Fourier hộp↔sinc ở trên, chỉ là đối xứng lại: vì mặt nạ của bộ lọc lý tưởng là một hộp cứng trong miền tần số, kernel không gian của nó là một hàm sinc (mà như đã thấy ở trên, có gợn sóng).

In [ ]:
def apply_frequency_filter(image, mask):
    F = np.fft.fftshift(np.fft.fft2(image)) # forward transform
    filtered = np.fft.ifft2(np.fft.ifftshift(F * mask)) # filter, then inverse transform
    return np.real(filtered)

low_pass = apply_frequency_filter(photo, low_pass_mask)
high_pass = apply_frequency_filter(photo, high_pass_mask)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [photo, low_pass, high_pass],
                          ['Original', 'Ideal low-pass\n(blurred)', 'Ideal high-pass\n(edges only)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Ảnh động này cho thấy hiệu ứng của việc thay đổi bán kính cắt của bộ lọc thông thấp lý tưởng, với ảnh đã lọc cập nhật trực tiếp. Với bán kính đủ lớn, mọi tần số đều đi qua, và ảnh không thay đổi. Khi bán kính thu nhỏ, ngày càng nhiều tần số bị loại bỏ hoàn toàn, và hiện tượng rung nêu trên ngày càng thấy rõ.

<img src="../animations/lowpass_cutoff_demo.gif" alt="Ảnh động bộ lọc thông thấp lý tưởng quét bán kính cắt từ lớn đến nhỏ rồi quay lại, cho thấy ảnh đã lọc ngày càng mờ hơn với hiện tượng rung thấy rõ gần các cạnh sắc, rồi sắc nét trở lại" style="display: block; margin: 0 auto; max-width: 680px;">

### Tránh hiện tượng rung bằng mặt nạ Gauss

Như đã giải thích, các gợn sóng mờ tỏa ra từ các cạnh của hình — hiện tượng Gibbs — gây ra bởi ngưỡng cắt bật/tắt cứng trong miền tần số của bộ lọc lý tưởng. Một mặt nạ Gauss, vốn giảm dần một cách mượt mà thay vì cắt đột ngột, tránh được hoàn toàn điều này.

In [ ]:
def gaussian_mask(shape, sigma):
    h, w = shape
    yy, xx = np.mgrid[:h, :w]
    dist2 = (yy - h / 2)**2 + (xx - w / 2)**2
    return np.exp(-dist2 / (2 * sigma**2))

smooth_low_pass = apply_frequency_filter(photo, gaussian_mask(photo.shape, sigma=15))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(low_pass, cmap='gray')
axes[0].set_title('Ideal (sharp cutoff)\nvisible ringing near edges')
axes[1].imshow(smooth_low_pass, cmap='gray')
axes[1].set_title('Gaussian mask\nsmooth falloff, no ringing')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Định lý tích chập

Tại sao lọc trong miền tần số lại hiệu quả? **Định lý tích chập**

$$\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}$$

phát biểu rằng tích chập hai tín hiệu trong miền không gian *chính xác* giống như nhân từng phần tử các biến đổi Fourier của chúng, rồi biến đổi ngược trở lại &mdash; tích chập trở thành phép nhân, và ngược lại.

### Một điểm vướng: DFT cho ra tích chập *vòng*

DFT ngầm coi ảnh như thể nó lặp lại vô hạn (hãy hình dung các viên gạch lát liền mạch kéo dài mãi theo cả hai hướng). Không có "bên ngoài ảnh", chỉ có "bản sao tiếp theo của cùng ảnh đó". Vì vậy nhân hai DFT rồi biến đổi ngược không cho ra tích chập thông thường ("tuyến tính") như ở Bài 10, nơi kernel cuối cùng trượt ra khỏi cạnh vào một vùng biên tường minh nào đó (số 0, sao chép, phản xạ); thay vào đó, nó cho ra **tích chập vòng (circular convolution)**, nơi kernel trượt ra khỏi cạnh phải sẽ quay vòng lại và lấy các điểm ảnh từ cạnh trái. Hai cách chỉ trùng nhau nếu cách xử lý biên ở phía không gian *cũng* là quay vòng (`cv2.BORDER_WRAP`).

In [ ]:
rng = np.random.default_rng(0)
test_img = rng.integers(0, 255, (64, 64)).astype(np.float64)

sigma, ksize = 3, 15
ax = np.arange(ksize) - ksize // 2
gx, gy = np.meshgrid(ax, ax)
kernel = np.exp(-(gx**2 + gy**2) / (2 * sigma**2))
kernel /= kernel.sum()

# Spatial convolution, wrap-around border to match the FFT's implicit circular convolution
spatial_result = cv2.filter2D(test_img, cv2.CV_64F, kernel, borderType=cv2.BORDER_WRAP)

# Frequency-domain: pad the kernel to image size, center it at (0,0) via np.roll, then multiply spectra
kernel_padded = np.zeros_like(test_img)
kernel_padded[:ksize, :ksize] = kernel
kernel_padded = np.roll(kernel_padded, -(ksize // 2), axis=0)
kernel_padded = np.roll(kernel_padded, -(ksize // 2), axis=1)

freq_result = np.real(np.fft.ifft2(np.fft.fft2(test_img) * np.fft.fft2(kernel_padded)))

print(f'max abs difference between spatial and frequency-domain convolution: '
      f'{np.abs(spatial_result - freq_result).max():.2e}')

Tuy nhiên, trên thực tế, lọc trong miền tần số hiếm khi được dùng trong thị giác máy tính thường ngày. Hầu hết các kernel &mdash; lọc hộp, Gauss, Sobel, làm sắc nét &mdash; đều nhỏ (3x3 hoặc 5x5), và tích chập không gian trực tiếp đã nhanh ở các kích thước đó, nên chi phí cố định của hai lần FFT xuôi và một lần FFT ngược thường không đáng bỏ ra. DFT chủ yếu phát huy giá trị khi kernel trở nên lớn.

### Bài tập

1. Xoay vân sóng trong thí nghiệm đỉnh phổ (làm cho các sọc chéo thay vì dọc) bằng cách xây dựng nó như một hàm của `x*cos(theta) + y*sin(theta)`. Hãy dự đoán, rồi kiểm chứng, đỉnh phổ di chuyển đến đâu.
2. Xây dựng một mặt nạ tần số **thông dải (band-pass)** (một hình vành khuyên: giữ các tần số giữa bán kính trong và bán kính ngoài, chặn mọi thứ khác) và áp dụng nó lên `photo`. Loại nội dung ảnh nào còn lại?
3. Thử `apply_frequency_filter` với sigma mặt nạ Gauss *rất* nhỏ (ví dụ 3) so với sigma lớn (ví dụ 60) trên một ảnh thử nghiệm giống ảnh chụp, và mô tả xu hướng khi sigma tăng dần về kích thước ảnh.
4. Lấy tích chập *tuyến tính* (không phải vòng) từ FFT: đệm số 0 cho cả `test_img` và `kernel` tới ít nhất `(image_size + kernel_size - 1)` theo mỗi trục trước khi biến đổi, nhân các phổ, biến đổi ngược, rồi cắt về kích thước ban đầu. So sánh kết quả với `cv2.filter2D(..., borderType=cv2.BORDER_CONSTANT)` (biên đệm số 0) thay vì `BORDER_WRAP`.
5. Dùng `%timeit`, đo tích chập trực tiếp (`cv2.filter2D`) so với tích chập dựa trên FFT (như `freq_result` ở trên) với các kích thước kernel Gauss 5, 15, 31, 63 và 127 trên ảnh 256x256. Khoảng từ kích thước kernel nào thì cách dùng FFT bắt đầu thắng thế?